In [ ]:
from pathlib import Path

In [ ]:
# Máximo índice de los ficheros fake de farmia que se cargan en landing
MAX_INDEX_FILES = 2

In [ ]:
class FarmiaDataset:
    def __init__(self, uri, catalog_name, schema_name, landing, lakehouse):
        self.uri = uri
        self.catalog = catalog_name
        self.schema = schema_name
        self.landing = landing
        self.lakehouse = lakehouse

    def download_dataset(self):
        source = self.uri
        target = self.landing_volume

        dbutils.fs.mkdirs(f"{target}/staging")
        dbutils.fs.mkdirs(f"{target}/_files")
        files = dbutils.fs.ls(source)

        for f in files:
            source_path = f"{source}/{f.name}"
            target_path = f"{target}/_files/{f.name}"

            try:
                dbutils.fs.ls(target_path)
            except Exception:
                print(f"Copiando {f.name}...")
                dbutils.fs.cp(source_path, target_path, True)

    def create_database(self):
        spark.sql(f"USE CATALOG {self.catalog}")
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {self.schema}")
        spark.sql(f"USE SCHEMA {self.schema}")

        self.__configure_directories()

    def __configure_directories(self):
        root_volume_name = f"{self.catalog}.{self.schema}"

        # Nombre de los volumes
        landing_name = f"{root_volume_name}.landing"
        meta_name = f"{root_volume_name}._meta"

        # Rutas lógicas
        self.landing_volume = f"/Volumes/{self.catalog}/{self.schema}/landing"
        self.checkpoint_volume = f"/Volumes/{self.catalog}/{self.schema}/_meta/_checkpoint"
        self.schema_volume = f"/Volumes/{self.catalog}/{self.schema}/_meta/_schema"

        # Rutas físicas ADLS
        landing_location = f"{self.landing}"
        meta_location = f"{self.lakehouse}/_meta"

        spark.sql(f"""
            CREATE EXTERNAL VOLUME IF NOT EXISTS {landing_name}
            LOCATION '{landing_location}'
        """)

        spark.sql(f"""
            CREATE EXTERNAL VOLUME IF NOT EXISTS {meta_name}
            LOCATION '{meta_location}'
        """)

    def clean_up(self):
        print("Borrando Schema, landing y _meta.")
        spark.sql(f"DROP SCHEMA IF EXISTS {self.schema} CASCADE")
        print("Ok")

    def __get_files_index(self, dir):
        files = dbutils.fs.ls(dir)
        if files:
            max_index = max({int(Path(f.name).stem[-2:]) for f in files})
            return max_index + 1
        return 0

    def __load_data(self, files_dir, staging_dir):
        index = self.__get_files_index(staging_dir)
        if index > MAX_INDEX_FILES:
            print("No hay archivos nuevos en el landing.")
            return 0

        file_list = [
            "app_events_{}.json",
            "inventory_{}.parquet",
            "orders_cdc_{}.parquet",
            "product_catalog_{}.parquet",
            "sensors_{}.json",
            "weather_{}.json"
        ]
        for f in file_list:
            file_name = f.format(f"{index:02}")
            print(f"Copiando {file_name} a staging...")
            dbutils.fs.cp(f"{files_dir}/{file_name}", f"{staging_dir}/{file_name}", True)

        return 1

    def load_data(self, num_files=1):
        files_dir = f"{self.landing_volume}/_files"
        staging_dir = f"{self.landing_volume}/staging"

        for _ in range(num_files):
            self.__load_data(files_dir, staging_dir)

In [ ]:
landing = "abfss://landing@masteravdc001sta.dfs.core.windows.net/farmia"
lakehouse = "abfss://lakehouse@masteravdc001sta.dfs.core.windows.net/farmia/bronze"
catalog = spark.sql("SELECT current_catalog()").collect()[0][0]

farmia_dataset = FarmiaDataset(
    "s3://ucm-lakehouses/files/",
    catalog,
    "farmia_bronze",
    landing,
    lakehouse
)

farmia_dataset.create_database()
farmia_dataset.download_dataset()